# SWAN-SF 5-Feature Train/Test Standardization (v3) with Metadata Sidecars

This notebook creates model-ready train/test arrays from the cleaned SWAN-SF tensor partitions while preserving aligned metadata sidecars.

**Split**

- Train: partitions **1, 2, 3, and 5**
- Test: partition **4**

**Model features**

- `TOTUSJH`
- `TOTBSQ`
- `TOTPOT`
- `TOTUSJZ`
- `ABSNJZH`

**Metadata sidecars**

- `partition_id`
- `HARPNUM`
- full per-case `timestamps` sequence
- observation-window `start_time` and `end_time`
- `source_file`

The metadata is saved separately and is **not included in the model feature tensor or standardized**. Every sidecar row remains aligned with the same row in `X` and `y`.

The notebook keeps the original memory-efficient logic:

1. Validate one partition at a time.
2. Locate the five requested feature channels from each partition's feature-column JSON.
3. Fit feature-wise mean and standard deviation using only train partitions.
4. Standardize and write arrays in chunks.
5. Save directly in aeon format: `(n_cases, n_channels, n_timepoints)`.
6. Write NumPy metadata sidecars during the same pass.
7. Build the readable metadata CSV from the completed sidecars in chunks, ensuring it cannot omit a partition.
8. Automatically rebuild a missing, malformed, or incomplete metadata CSV during final validation.

**v3 fixes**

- Mixed timestamp formats are parsed with `format="mixed"` when supported.
- Missing or malformed start/end times are recovered from the timestamps encoded in `source_file`.
- Metadata CSVs are generated from the completed `.npy` sidecars instead of being appended partition-by-partition.
- Final validation automatically repairs an incomplete CSV and reads it with explicit dtypes and `low_memory=False`.


## 1. Imports and paths

In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


In [ ]:
# Mount Google Drive in Colab.
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# -----------------------------
# User configuration
# -----------------------------

BASE_DIR = Path("/content/drive/MyDrive/solar_flare_forecasting")
DATA_DIR = BASE_DIR / "Data" / "processed_tensors"

TRAIN_PARTITIONS = [1, 2, 3, 5]
TEST_PARTITIONS = [4]

SELECTED_FEATURES = [
    "TOTUSJH",
    "TOTBSQ",
    "TOTPOT",
    "TOTUSJZ",
    "ABSNJZH",
]

X_KEY = "X"
Y_KEY = "y_flare"

# Lower this if Colab memory is tight.
CHUNK_SIZE = 4096

# Chunk size used when generating readable metadata CSV files.
CSV_CHUNK_SIZE = 50_000

# SWAN-SF cadence: 60 timepoints at 12-minute spacing.
TIME_STEP_MINUTES = 12

# Output folder for the five-feature standardized dataset and sidecars.
OUT_DIR = BASE_DIR / "Data"/ "5_features_standardized"
OUT_DIR.mkdir(parents=True, exist_ok=True)

partition_paths = {
    p: DATA_DIR / f"partition{p}_combined_clean.npz"
    for p in range(1, 6)
}

feature_column_paths = {
    p: DATA_DIR / f"partition{p}_feature_columns.json"
    for p in range(1, 6)
}

metadata_clean_paths = {
    p: DATA_DIR / f"partition{p}_metadata_clean.csv"
    for p in range(1, 6)
}

metadata_all_paths = {
    p: DATA_DIR / f"partition{p}_metadata_all.csv"
    for p in range(1, 6)
}

clean_index_paths = {
    p: DATA_DIR / f"partition{p}_clean_indices.npy"
    for p in range(1, 6)
}

print("Input directory:", DATA_DIR)
print("Output directory:", OUT_DIR)


## 2. Helper functions

In [ ]:
def memory_gb_from_shape(shape, dtype=np.float32):
    return np.prod(shape) * np.dtype(dtype).itemsize / (1024 ** 3)


def class_count_dict(y):
    values, counts = np.unique(y, return_counts=True)
    return {str(v): int(c) for v, c in zip(values, counts)}


def inspect_npz(path):
    path = Path(path)
    print(f"File: {path.name}")
    with np.load(path, allow_pickle=True) as data:
        print("Keys:", list(data.files))
        for key in data.files:
            arr = data[key]
            print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")


def load_feature_names(path):
    """Load feature names from either a JSON list or a simple JSON dictionary."""
    with open(path, "r") as f:
        payload = json.load(f)

    if isinstance(payload, list):
        names = payload
    elif isinstance(payload, dict):
        for key in ("feature_columns", "features", "columns"):
            if key in payload and isinstance(payload[key], list):
                names = payload[key]
                break
        else:
            # Accept dictionaries such as {"0": "TOTUSJH", ...}.
            try:
                names = [payload[k] for k in sorted(payload, key=lambda x: int(x))]
            except (ValueError, TypeError):
                raise ValueError(
                    f"Could not identify the feature-name list in {path.name}."
                )
    else:
        raise TypeError(f"Unsupported JSON structure in {path.name}: {type(payload)}")

    return [str(name) for name in names]


def resolve_column(df, canonical_name, candidates):
    """Find a metadata column using case-insensitive candidate matching."""
    lower_to_original = {str(col).lower(): col for col in df.columns}

    for candidate in [canonical_name] + list(candidates):
        if candidate in df.columns:
            return candidate
        match = lower_to_original.get(candidate.lower())
        if match is not None:
            return match

    raise KeyError(
        f"Could not find metadata field '{canonical_name}'. "
        f"Tried {([canonical_name] + list(candidates))}. "
        f"Available columns: {list(df.columns)}"
    )


def parse_metadata_datetimes(values, source_files, which, partition_id):
    """Parse mixed timestamp formats and recover missing values from source filenames.

    The metadata CSVs can contain a mixture of values such as
    ``2011-02-16 23:00:00`` and ``2011-02-16T23:00:00``. Recent pandas versions
    may infer one format from the first row and convert the other valid format to NaT.
    ``format="mixed"`` parses each value according to its own format.
    """
    text = values.astype("string").str.strip()
    text = text.replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA})

    try:
        parsed = pd.to_datetime(
            text,
            format="mixed",
            errors="coerce",
            utc=True,
        )
    except (TypeError, ValueError):
        # Compatibility fallback for older pandas releases without format="mixed".
        parsed = pd.to_datetime(text, errors="coerce", utc=True)

    # Store timezone-naive UTC values so NumPy datetime64 sidecars can be written.
    parsed = parsed.dt.tz_convert(None)

    # The source filename contains authoritative observation-window timestamps:
    # ..._s2011-02-16T23:00:00_e2011-02-17T10:48:00.csv
    source_text = source_files.astype("string").str.strip()
    extracted = source_text.str.extract(
        r"_s(?P<start>\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})"
        r"_e(?P<end>\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})(?:\.csv)?$",
        expand=True,
    )
    filename_values = extracted["start" if which == "start" else "end"]
    filename_parsed = pd.to_datetime(
        filename_values,
        format="%Y-%m-%dT%H:%M:%S",
        errors="coerce",
        utc=True,
    ).dt.tz_convert(None)

    parsed = parsed.fillna(filename_parsed)

    if parsed.isna().any():
        bad_mask = parsed.isna()
        examples = pd.DataFrame({
            "raw_value": text[bad_mask],
            "source_file": source_text[bad_mask],
        }).head(10)
        raise ValueError(
            f"Partition {partition_id}: {int(bad_mask.sum())} invalid {which}_time "
            "values remain after mixed-format parsing and filename recovery. "
            f"Examples:\n{examples.to_string(index=False)}"
        )

    return parsed


def normalize_metadata_columns(metadata, partition_id):
    """Return only the requested aligned metadata using consistent output names."""
    metadata = metadata.copy()
    metadata = metadata.loc[:, ~metadata.columns.astype(str).str.startswith("Unnamed:")]

    harp_col = resolve_column(
        metadata,
        "HARPNUM",
        ["harpnum", "harp_num", "HARP_NUM"],
    )
    start_col = resolve_column(
        metadata,
        "start_time",
        ["START_TIME", "window_start", "start_timestamp", "timestamp_start"],
    )
    end_col = resolve_column(
        metadata,
        "end_time",
        ["END_TIME", "window_end", "end_timestamp", "timestamp_end"],
    )
    source_col = resolve_column(
        metadata,
        "source_file",
        ["SOURCE_FILE", "relative_path", "file", "filename"],
    )

    source_files = metadata[source_col].astype("string")
    start_times = parse_metadata_datetimes(
        metadata[start_col], source_files, "start", partition_id
    )
    end_times = parse_metadata_datetimes(
        metadata[end_col], source_files, "end", partition_id
    )

    normalized = pd.DataFrame({
        "partition_id": np.full(len(metadata), int(partition_id), dtype=np.int8),
        "HARPNUM": pd.to_numeric(metadata[harp_col], errors="coerce").astype("Int64"),
        "start_time": start_times,
        "end_time": end_times,
        "source_file": source_files,
    })

    if normalized["HARPNUM"].isna().any():
        bad = int(normalized["HARPNUM"].isna().sum())
        raise ValueError(f"Partition {partition_id}: {bad} rows have invalid HARPNUM values.")

    if normalized["source_file"].isna().any():
        bad = int(normalized["source_file"].isna().sum())
        raise ValueError(f"Partition {partition_id}: {bad} rows have missing source_file values.")

    normalized["HARPNUM"] = normalized["HARPNUM"].astype(np.int64)
    normalized["source_file"] = normalized["source_file"].astype(str)

    return normalized


def load_aligned_metadata(partition_id):
    """Load metadata already filtered to the clean tensor rows.

    Preferred source:
        partitionN_metadata_clean.csv

    Fallback:
        partitionN_metadata_all.csv selected by partitionN_clean_indices.npy
    """
    clean_path = metadata_clean_paths[partition_id]
    all_path = metadata_all_paths[partition_id]
    index_path = clean_index_paths[partition_id]

    if clean_path.exists():
        raw_metadata = pd.read_csv(clean_path, low_memory=False)
        source_used = clean_path.name
    elif all_path.exists() and index_path.exists():
        raw_metadata = pd.read_csv(all_path, low_memory=False)
        clean_indices = np.load(index_path)

        if clean_indices.dtype == bool:
            if len(clean_indices) != len(raw_metadata):
                raise ValueError(
                    f"Partition {partition_id}: Boolean clean-index mask length "
                    f"{len(clean_indices)} does not match metadata length {len(raw_metadata)}."
                )
            raw_metadata = raw_metadata.loc[clean_indices].reset_index(drop=True)
        else:
            raw_metadata = raw_metadata.iloc[clean_indices.astype(np.int64)].reset_index(drop=True)

        source_used = f"{all_path.name} + {index_path.name}"
    else:
        raise FileNotFoundError(
            f"Partition {partition_id}: expected either {clean_path.name}, or both "
            f"{all_path.name} and {index_path.name}."
        )

    normalized = normalize_metadata_columns(raw_metadata, partition_id)
    return normalized, source_used


def validate_metadata_against_tensor(metadata, X, y, partition_id, raw_metadata=None):
    if len(metadata) != X.shape[0]:
        raise ValueError(
            f"Partition {partition_id}: metadata rows ({len(metadata)}) do not match "
            f"tensor cases ({X.shape[0]})."
        )

    # Optional label-order check when the original metadata has a y_flare column.
    if raw_metadata is not None:
        label_candidates = [c for c in raw_metadata.columns if str(c).lower() == "y_flare"]
        if label_candidates:
            metadata_y = pd.to_numeric(
                raw_metadata[label_candidates[0]], errors="coerce"
            ).to_numpy()
            if len(metadata_y) == len(y) and not np.isnan(metadata_y).any():
                if not np.array_equal(metadata_y.astype(np.int8), y.astype(np.int8)):
                    raise ValueError(
                        f"Partition {partition_id}: metadata y_flare order does not match tensor y."
                    )


def update_running_feature_sums(
    X,
    selected_indices,
    feature_sum,
    feature_sumsq,
    total_observations,
    chunk_size=CHUNK_SIZE,
):
    """Update train-only sums for the selected features.

    X format: (n_cases, n_timepoints, n_all_features)
    """
    n_cases = X.shape[0]

    for start in range(0, n_cases, chunk_size):
        end = min(start + chunk_size, n_cases)
        chunk = X[start:end, :, selected_indices]

        feature_sum += chunk.sum(axis=(0, 1), dtype=np.float64)
        feature_sumsq += np.square(chunk, dtype=np.float64).sum(axis=(0, 1))
        total_observations += chunk.shape[0] * chunk.shape[1]

    return feature_sum, feature_sumsq, total_observations


def validate_saved_aeon_array(path, chunk_size=CHUNK_SIZE):
    X = np.load(path, mmap_mode="r")
    n_cases, n_channels, n_timepoints = X.shape

    nan_count = 0
    inf_count = 0
    feature_sum = np.zeros(n_channels, dtype=np.float64)
    feature_sumsq = np.zeros(n_channels, dtype=np.float64)
    total_observations = 0

    for start in tqdm(
        range(0, n_cases, chunk_size),
        desc=f"Validating {Path(path).name}",
    ):
        end = min(start + chunk_size, n_cases)
        chunk = X[start:end]

        nan_count += int(np.isnan(chunk).sum())
        inf_count += int(np.isinf(chunk).sum())
        feature_sum += chunk.sum(axis=(0, 2), dtype=np.float64)
        feature_sumsq += np.square(chunk, dtype=np.float64).sum(axis=(0, 2))
        total_observations += chunk.shape[0] * chunk.shape[2]

    mean = feature_sum / total_observations
    variance = np.maximum((feature_sumsq / total_observations) - (mean ** 2), 0.0)
    std = np.sqrt(variance)

    return {
        "shape": tuple(int(v) for v in X.shape),
        "dtype": str(X.dtype),
        "memory_GB": round(memory_gb_from_shape(X.shape, X.dtype), 3),
        "NaN_count": int(nan_count),
        "Inf_count": int(inf_count),
        "feature_means": {
            feature: float(value)
            for feature, value in zip(SELECTED_FEATURES, mean)
        },
        "feature_stds": {
            feature: float(value)
            for feature, value in zip(SELECTED_FEATURES, std)
        },
    }


## 3. Inspect tensor keys and select the five feature channels

The feature indices are read from `partitionN_feature_columns.json`; they are not hard-coded. This prevents accidental use of the wrong channels if the original tensor column order changes.


In [ ]:
inspect_npz(partition_paths[1])

reference_feature_names = load_feature_names(feature_column_paths[1])
missing_features = [
    feature for feature in SELECTED_FEATURES
    if feature not in reference_feature_names
]

if missing_features:
    raise ValueError(
        f"Requested features are missing from partition 1: {missing_features}"
    )

selected_feature_indices = np.array(
    [reference_feature_names.index(feature) for feature in SELECTED_FEATURES],
    dtype=np.int64,
)

print("Total source features:", len(reference_feature_names))
print("Selected features and source indices:")
for feature, index in zip(SELECTED_FEATURES, selected_feature_indices):
    print(f"  {feature}: index {index}")


In [ ]:
# Confirm that all five partitions use the same feature order.
for p in range(1, 6):
    names = load_feature_names(feature_column_paths[p])
    if names != reference_feature_names:
        raise ValueError(
            f"Partition {p} feature-column order differs from partition 1."
        )

print("All partition feature-column JSON files use the same order.")


## 4. Validate raw tensors and aligned metadata

For each partition this checks:

- `X` and `y` contain the same number of cases
- the tensor's feature dimension matches the feature JSON
- no NaN or infinite values exist in the five selected channels
- cleaned metadata has exactly one row per tensor case
- requested metadata fields are present and parse correctly


In [ ]:
partition_summaries = []

for p in tqdm(TRAIN_PARTITIONS + TEST_PARTITIONS, desc="Validating partitions"):
    with np.load(partition_paths[p], allow_pickle=True) as data:
        X = data[X_KEY]
        y = data[Y_KEY]

        metadata, metadata_source = load_aligned_metadata(p)

        if X.shape[0] != y.shape[0]:
            raise ValueError(
                f"Partition {p}: X has {X.shape[0]} cases but y has {y.shape[0]}."
            )
        if X.ndim != 3:
            raise ValueError(f"Partition {p}: expected a 3D tensor, found shape {X.shape}.")
        if X.shape[2] != len(reference_feature_names):
            raise ValueError(
                f"Partition {p}: tensor has {X.shape[2]} features but JSON lists "
                f"{len(reference_feature_names)}."
            )
        if len(metadata) != X.shape[0]:
            raise ValueError(
                f"Partition {p}: metadata has {len(metadata)} rows but X has "
                f"{X.shape[0]} cases."
            )

        X_selected = X[:, :, selected_feature_indices]
        nan_count = int(np.isnan(X_selected).sum())
        inf_count = int(np.isinf(X_selected).sum())

        if nan_count != 0:
            raise ValueError(
                f"Partition {p}: selected features contain {nan_count} NaN values."
            )
        if inf_count != 0:
            raise ValueError(
                f"Partition {p}: selected features contain {inf_count} infinite values."
            )

        summary = {
            "partition": int(p),
            "split": "train" if p in TRAIN_PARTITIONS else "test",
            "file": partition_paths[p].name,
            "metadata_source": metadata_source,
            "X_shape_full": tuple(int(v) for v in X.shape),
            "X_shape_selected": (
                int(X.shape[0]),
                int(X.shape[1]),
                len(SELECTED_FEATURES),
            ),
            "y_shape": tuple(int(v) for v in y.shape),
            "X_dtype": str(X.dtype),
            "y_dtype": str(y.dtype),
            "selected_X_memory_GB": round(
                memory_gb_from_shape(
                    (X.shape[0], X.shape[1], len(SELECTED_FEATURES)),
                    X.dtype,
                ),
                3,
            ),
            "NaN_count_selected": nan_count,
            "Inf_count_selected": inf_count,
            "class_counts": class_count_dict(y),
            "metadata_rows": int(len(metadata)),
            "unique_HARPNUM": int(metadata["HARPNUM"].nunique()),
            "start_time_min": metadata["start_time"].min().isoformat(),
            "end_time_max": metadata["end_time"].max().isoformat(),
            "max_source_file_length": int(metadata["source_file"].str.len().max()),
        }
        partition_summaries.append(summary)

    del X, y, X_selected, metadata
    gc.collect()

summary_df = pd.DataFrame(partition_summaries)
summary_df


In [ ]:
# Validate common time-series shape and calculate split sizes.
reference_shape = partition_summaries[0]["X_shape_full"]
n_timepoints = int(reference_shape[1])
n_selected_features = len(SELECTED_FEATURES)

for row in partition_summaries:
    if row["X_shape_full"][1] != n_timepoints:
        raise ValueError(
            f"Partition {row['partition']}: timepoint count differs from partition 1."
        )

n_train = int(
    sum(row["X_shape_full"][0] for row in partition_summaries if row["split"] == "train")
)
n_test = int(
    sum(row["X_shape_full"][0] for row in partition_summaries if row["split"] == "test")
)

max_source_file_length = max(
    row["max_source_file_length"] for row in partition_summaries
)
# Ensure a valid Unicode dtype even if filenames happen to be empty.
max_source_file_length = max(1, int(max_source_file_length))

print("n_train:", n_train)
print("n_test:", n_test)
print("n_timepoints:", n_timepoints)
print("selected feature count:", n_selected_features)
print("aeon train shape:", (n_train, n_selected_features, n_timepoints))
print("aeon test shape:", (n_test, n_selected_features, n_timepoints))
print("maximum source_file string length:", max_source_file_length)


In [ ]:
# Combined class counts without concatenating labels.
combined_class_counts = {"train": {}, "test": {}}

for row in partition_summaries:
    split = row["split"]
    for label, count in row["class_counts"].items():
        combined_class_counts[split][label] = (
            combined_class_counts[split].get(label, 0) + int(count)
        )

print(json.dumps(combined_class_counts, indent=2))


## 5. Fit the five-feature scaler on train partitions only

For each selected feature:

```text
mean_j = mean of X_train[:, :, j]
std_j  = std  of X_train[:, :, j]
```

The calculation uses partitions 1, 2, 3, and 5 only. Partition 4 is never used to fit the scaler.


In [ ]:
feature_sum = np.zeros(n_selected_features, dtype=np.float64)
feature_sumsq = np.zeros(n_selected_features, dtype=np.float64)
total_observations = 0

for p in tqdm(TRAIN_PARTITIONS, desc="First pass: fitting train scaler"):
    with np.load(partition_paths[p], allow_pickle=True) as data:
        X = data[X_KEY]
        feature_sum, feature_sumsq, total_observations = update_running_feature_sums(
            X=X,
            selected_indices=selected_feature_indices,
            feature_sum=feature_sum,
            feature_sumsq=feature_sumsq,
            total_observations=total_observations,
            chunk_size=CHUNK_SIZE,
        )

    del X
    gc.collect()

feature_mean_1d = feature_sum / total_observations
feature_var_1d = np.maximum(
    (feature_sumsq / total_observations) - (feature_mean_1d ** 2),
    0.0,
)
feature_std_1d = np.sqrt(feature_var_1d)

epsilon = 1e-8
zero_std_mask = feature_std_1d < epsilon
feature_std_safe_1d = feature_std_1d.copy()
feature_std_safe_1d[zero_std_mask] = 1.0

feature_mean = feature_mean_1d.astype(np.float32).reshape(
    1, 1, n_selected_features
)
feature_std = feature_std_1d.astype(np.float32).reshape(
    1, 1, n_selected_features
)
feature_std_safe = feature_std_safe_1d.astype(np.float32).reshape(
    1, 1, n_selected_features
)

scaler_table = pd.DataFrame({
    "feature": SELECTED_FEATURES,
    "source_index": selected_feature_indices,
    "train_mean": feature_mean_1d,
    "train_std": feature_std_1d,
    "zero_std": zero_std_mask,
})
scaler_table


In [ ]:
scaler_path = OUT_DIR / "scaler_params_5_features.npz"
feature_names_path = OUT_DIR / "selected_feature_names.json"

np.savez(
    scaler_path,
    selected_features=np.array(SELECTED_FEATURES),
    selected_feature_indices=selected_feature_indices,
    feature_mean=feature_mean,
    feature_std=feature_std,
    feature_std_safe=feature_std_safe,
    feature_mean_1d=feature_mean_1d.astype(np.float32),
    feature_std_1d=feature_std_1d.astype(np.float32),
    zero_std_mask=zero_std_mask,
    train_partitions=np.array(TRAIN_PARTITIONS, dtype=np.int16),
    test_partitions=np.array(TEST_PARTITIONS, dtype=np.int16),
)

with open(feature_names_path, "w") as f:
    json.dump(SELECTED_FEATURES, f, indent=2)

print("Saved scaler:", scaler_path)
print("Saved selected feature names:", feature_names_path)


## 6. Standardize, transpose, and save metadata sidecars

The feature tensors, labels, and NumPy metadata sidecars are written directly to disk in aligned order. After those arrays are flushed, the readable metadata CSV is rebuilt from the completed sidecars in chunks. This avoids partial CSV output if a large partition-level append is interrupted or truncated by Google Drive.


In [ ]:
X_train_out_path = OUT_DIR / "X_train_standardized_aeon.npy"
y_train_out_path = OUT_DIR / "y_train.npy"
X_test_out_path = OUT_DIR / "X_test_standardized_aeon.npy"
y_test_out_path = OUT_DIR / "y_test.npy"

train_metadata_csv_path = OUT_DIR / "train_metadata.csv"
test_metadata_csv_path = OUT_DIR / "test_metadata.csv"

train_sidecar_paths = {
    "partition_id": OUT_DIR / "train_partition_id.npy",
    "HARPNUM": OUT_DIR / "train_HARPNUM.npy",
    "timestamps": OUT_DIR / "train_timestamps.npy",
    "start_time": OUT_DIR / "train_start_time.npy",
    "end_time": OUT_DIR / "train_end_time.npy",
    "source_file": OUT_DIR / "train_source_file.npy",
}

test_sidecar_paths = {
    "partition_id": OUT_DIR / "test_partition_id.npy",
    "HARPNUM": OUT_DIR / "test_HARPNUM.npy",
    "timestamps": OUT_DIR / "test_timestamps.npy",
    "start_time": OUT_DIR / "test_start_time.npy",
    "end_time": OUT_DIR / "test_end_time.npy",
    "source_file": OUT_DIR / "test_source_file.npy",
}

print("Train X:", X_train_out_path)
print("Train metadata CSV:", train_metadata_csv_path)
print("Test X:", X_test_out_path)
print("Test metadata CSV:", test_metadata_csv_path)


In [ ]:
def write_metadata_csv_from_sidecars(
    metadata_csv_path,
    sidecar_paths,
    expected_cases,
    chunk_size=CSV_CHUNK_SIZE,
):
    """Build a readable metadata CSV from completed NumPy sidecars.

    The NumPy sidecars are the alignment source of truth. Constructing the CSV
    from them after all arrays are flushed guarantees the CSV has the same row
    count and ordering as X and y.
    """
    metadata_csv_path = Path(metadata_csv_path)

    partition_id = np.load(sidecar_paths["partition_id"], mmap_mode="r")
    harpnum = np.load(sidecar_paths["HARPNUM"], mmap_mode="r")
    start_time = np.load(sidecar_paths["start_time"], mmap_mode="r")
    end_time = np.load(sidecar_paths["end_time"], mmap_mode="r")
    source_file = np.load(sidecar_paths["source_file"], mmap_mode="r")

    lengths = {
        "partition_id": len(partition_id),
        "HARPNUM": len(harpnum),
        "start_time": len(start_time),
        "end_time": len(end_time),
        "source_file": len(source_file),
    }

    if any(length != expected_cases for length in lengths.values()):
        raise ValueError(
            "Cannot build metadata CSV because sidecar lengths are not aligned: "
            f"{lengths}; expected {expected_cases}."
        )

    if metadata_csv_path.exists():
        metadata_csv_path.unlink()

    wrote_header = False

    for start in tqdm(
        range(0, expected_cases, chunk_size),
        desc=f"Writing {metadata_csv_path.name}",
    ):
        end = min(start + chunk_size, expected_cases)

        chunk_df = pd.DataFrame({
            "partition_id": np.asarray(
                partition_id[start:end],
                dtype=np.int8,
            ),
            "HARPNUM": np.asarray(
                harpnum[start:end],
                dtype=np.int64,
            ),
            "start_time": np.asarray(
                start_time[start:end],
                dtype="datetime64[ns]",
            ),
            "end_time": np.asarray(
                end_time[start:end],
                dtype="datetime64[ns]",
            ),
            "source_file": np.asarray(
                source_file[start:end]
            ).astype(str),
        })

        chunk_df.to_csv(
            metadata_csv_path,
            mode="a",
            index=False,
            header=not wrote_header,
            date_format="%Y-%m-%dT%H:%M:%S",
        )
        wrote_header = True

        del chunk_df

    # Read only one small numeric column to confirm the exact saved row count.
    saved_row_count = len(
        pd.read_csv(
            metadata_csv_path,
            usecols=["partition_id"],
            dtype={"partition_id": "int8"},
            low_memory=False,
        )
    )

    if saved_row_count != expected_cases:
        raise ValueError(
            f"{metadata_csv_path.name}: wrote {saved_row_count} rows; "
            f"expected {expected_cases}."
        )

    del partition_id, harpnum, start_time, end_time, source_file
    gc.collect()

    print(
        f"Saved {metadata_csv_path.name}: "
        f"{saved_row_count:,} aligned rows"
    )


def write_split_to_disk(
    partitions,
    X_out_path,
    y_out_path,
    metadata_csv_path,
    sidecar_paths,
    total_cases,
    split_name,
):
    """Write standardized tensors, labels, and aligned sidecars in one pass."""

    X_out = np.lib.format.open_memmap(
        X_out_path,
        mode="w+",
        dtype=np.float32,
        shape=(total_cases, n_selected_features, n_timepoints),
    )
    y_out = np.lib.format.open_memmap(
        y_out_path,
        mode="w+",
        dtype=np.int8,
        shape=(total_cases,),
    )

    partition_out = np.lib.format.open_memmap(
        sidecar_paths["partition_id"],
        mode="w+",
        dtype=np.int8,
        shape=(total_cases,),
    )
    harpnum_out = np.lib.format.open_memmap(
        sidecar_paths["HARPNUM"],
        mode="w+",
        dtype=np.int64,
        shape=(total_cases,),
    )
    timestamps_out = np.lib.format.open_memmap(
        sidecar_paths["timestamps"],
        mode="w+",
        dtype="datetime64[ns]",
        shape=(total_cases, n_timepoints),
    )
    start_time_out = np.lib.format.open_memmap(
        sidecar_paths["start_time"],
        mode="w+",
        dtype="datetime64[ns]",
        shape=(total_cases,),
    )
    end_time_out = np.lib.format.open_memmap(
        sidecar_paths["end_time"],
        mode="w+",
        dtype="datetime64[ns]",
        shape=(total_cases,),
    )
    source_file_out = np.lib.format.open_memmap(
        sidecar_paths["source_file"],
        mode="w+",
        dtype=f"<U{max_source_file_length}",
        shape=(total_cases,),
    )

    write_index = 0

    for p in tqdm(partitions, desc=f"Second pass: writing {split_name}"):
        with np.load(partition_paths[p], allow_pickle=True) as data:
            X = data[X_KEY]
            y = data[Y_KEY]

            metadata, metadata_source = load_aligned_metadata(p)

            if len(metadata) != X.shape[0] or len(y) != X.shape[0]:
                raise ValueError(
                    f"Partition {p}: X, y, and metadata lengths are not aligned."
                )

            # Strong order check when y_flare exists in the cleaned metadata file.
            clean_path = metadata_clean_paths[p]
            if clean_path.exists():
                raw_metadata_for_check = pd.read_csv(
                    clean_path,
                    low_memory=False,
                )
                y_col = next(
                    (
                        col for col in raw_metadata_for_check.columns
                        if str(col).lower() == "y_flare"
                    ),
                    None,
                )
                if y_col is not None:
                    metadata_y = pd.to_numeric(
                        raw_metadata_for_check[y_col],
                        errors="coerce",
                    )
                    if metadata_y.notna().all():
                        if not np.array_equal(
                            metadata_y.to_numpy(dtype=np.int8),
                            y.astype(np.int8),
                        ):
                            raise ValueError(
                                f"Partition {p}: metadata y_flare does not match tensor y."
                            )
                del raw_metadata_for_check

            n_cases_partition = X.shape[0]

            for start in range(0, n_cases_partition, CHUNK_SIZE):
                end = min(start + CHUNK_SIZE, n_cases_partition)
                batch_size = end - start
                output_slice = slice(write_index, write_index + batch_size)

                chunk = X[start:end, :, selected_feature_indices]
                chunk_std = (chunk - feature_mean) / feature_std_safe
                chunk_aeon = np.transpose(
                    chunk_std,
                    (0, 2, 1),
                ).astype(np.float32, copy=False)

                X_out[output_slice] = chunk_aeon
                y_out[output_slice] = y[start:end].astype(np.int8, copy=False)

                metadata_chunk = metadata.iloc[start:end]
                partition_out[output_slice] = metadata_chunk[
                    "partition_id"
                ].to_numpy(dtype=np.int8)
                harpnum_out[output_slice] = metadata_chunk[
                    "HARPNUM"
                ].to_numpy(dtype=np.int64)
                start_values = metadata_chunk[
                    "start_time"
                ].to_numpy(dtype="datetime64[ns]")
                end_values = metadata_chunk[
                    "end_time"
                ].to_numpy(dtype="datetime64[ns]")

                time_offsets = (
                    np.arange(n_timepoints, dtype=np.int64)
                    * np.timedelta64(TIME_STEP_MINUTES, "m")
                )
                timestamp_matrix = start_values[:, None] + time_offsets[None, :]

                # The final timestamp should match metadata end_time exactly.
                if not np.array_equal(timestamp_matrix[:, -1], end_values):
                    mismatch_count = int(
                        np.count_nonzero(timestamp_matrix[:, -1] != end_values)
                    )
                    raise ValueError(
                        f"Partition {p}: {mismatch_count} rows have start/end times "
                        f"inconsistent with {n_timepoints} points at "
                        f"{TIME_STEP_MINUTES}-minute cadence."
                    )

                timestamps_out[output_slice] = timestamp_matrix
                start_time_out[output_slice] = start_values
                end_time_out[output_slice] = end_values
                source_file_out[output_slice] = metadata_chunk[
                    "source_file"
                ].to_numpy(dtype=str)

                write_index += batch_size

        del X, y, metadata
        gc.collect()

    if write_index != total_cases:
        raise ValueError(
            f"{split_name}: expected {total_cases} cases but wrote {write_index}."
        )

    arrays_to_flush = [
        X_out,
        y_out,
        partition_out,
        harpnum_out,
        timestamps_out,
        start_time_out,
        end_time_out,
        source_file_out,
    ]
    for array in arrays_to_flush:
        array.flush()

    del arrays_to_flush
    del X_out, y_out
    del partition_out, harpnum_out, timestamps_out, start_time_out, end_time_out, source_file_out
    gc.collect()

    # Build the readable CSV only after the complete sidecars are safely on disk.
    write_metadata_csv_from_sidecars(
        metadata_csv_path=metadata_csv_path,
        sidecar_paths=sidecar_paths,
        expected_cases=total_cases,
    )


write_split_to_disk(
    partitions=TRAIN_PARTITIONS,
    X_out_path=X_train_out_path,
    y_out_path=y_train_out_path,
    metadata_csv_path=train_metadata_csv_path,
    sidecar_paths=train_sidecar_paths,
    total_cases=n_train,
    split_name="train",
)

write_split_to_disk(
    partitions=TEST_PARTITIONS,
    X_out_path=X_test_out_path,
    y_out_path=y_test_out_path,
    metadata_csv_path=test_metadata_csv_path,
    sidecar_paths=test_sidecar_paths,
    total_cases=n_test,
    split_name="test",
)

print("Finished writing standardized arrays and metadata sidecars.")


## 7. Validate saved tensors, labels, and metadata alignment

The train feature means should be near 0 and train feature standard deviations near 1. Test statistics do not need to be exactly 0 and 1 because the scaler was fit only on train data.

Before checking alignment, the validation code verifies the metadata CSV. If the CSV is missing, malformed, or has the wrong row count, it is automatically rebuilt from the complete NumPy sidecars.


In [ ]:
train_saved_summary = validate_saved_aeon_array(X_train_out_path)
test_saved_summary = validate_saved_aeon_array(X_test_out_path)

print("TRAIN SAVED SUMMARY")
print(json.dumps(train_saved_summary, indent=2))

print("\nTEST SAVED SUMMARY")
print(json.dumps(test_saved_summary, indent=2))


In [ ]:
def read_metadata_csv(metadata_csv_path):
    """Read the final metadata CSV using explicit stable dtypes."""
    return pd.read_csv(
        metadata_csv_path,
        dtype={
            "partition_id": "int8",
            "HARPNUM": "int64",
            "source_file": "string",
        },
        parse_dates=["start_time", "end_time"],
        low_memory=False,
    )


def load_or_rebuild_metadata_csv(
    metadata_csv_path,
    sidecar_paths,
    expected_cases,
    split_name,
):
    """Return a valid metadata CSV, rebuilding it from sidecars when needed."""
    metadata_csv_path = Path(metadata_csv_path)
    rebuild_reason = None

    if not metadata_csv_path.exists():
        rebuild_reason = "file is missing"
    else:
        try:
            metadata_csv = read_metadata_csv(metadata_csv_path)
            if len(metadata_csv) != expected_cases:
                rebuild_reason = (
                    f"row count is {len(metadata_csv):,}; "
                    f"expected {expected_cases:,}"
                )
            else:
                return metadata_csv
        except (ValueError, TypeError, KeyError, pd.errors.ParserError) as exc:
            rebuild_reason = f"file could not be read consistently: {exc}"

    print(
        f"{split_name}: rebuilding {metadata_csv_path.name} because "
        f"{rebuild_reason}."
    )

    write_metadata_csv_from_sidecars(
        metadata_csv_path=metadata_csv_path,
        sidecar_paths=sidecar_paths,
        expected_cases=expected_cases,
    )

    metadata_csv = read_metadata_csv(metadata_csv_path)

    if len(metadata_csv) != expected_cases:
        raise ValueError(
            f"{split_name}: rebuilt metadata CSV has {len(metadata_csv):,} rows; "
            f"expected {expected_cases:,}."
        )

    return metadata_csv


def validate_saved_split(
    X_path,
    y_path,
    metadata_csv_path,
    sidecar_paths,
    expected_cases,
    expected_partitions,
    split_name,
):
    X = np.load(X_path, mmap_mode="r")
    y = np.load(y_path, mmap_mode="r")
    partition_id = np.load(sidecar_paths["partition_id"], mmap_mode="r")
    harpnum = np.load(sidecar_paths["HARPNUM"], mmap_mode="r")
    timestamps = np.load(sidecar_paths["timestamps"], mmap_mode="r")
    start_time = np.load(sidecar_paths["start_time"], mmap_mode="r")
    end_time = np.load(sidecar_paths["end_time"], mmap_mode="r")
    source_file = np.load(sidecar_paths["source_file"], mmap_mode="r")

    # Check the NumPy outputs before using them to repair a CSV.
    numpy_lengths = {
        "X": len(X),
        "y": len(y),
        "partition_id": len(partition_id),
        "HARPNUM": len(harpnum),
        "timestamps": len(timestamps),
        "start_time": len(start_time),
        "end_time": len(end_time),
        "source_file": len(source_file),
    }

    if any(length != expected_cases for length in numpy_lengths.values()):
        raise ValueError(
            f"{split_name}: NumPy outputs are not aligned: {numpy_lengths}"
        )

    metadata_csv = load_or_rebuild_metadata_csv(
        metadata_csv_path=metadata_csv_path,
        sidecar_paths=sidecar_paths,
        expected_cases=expected_cases,
        split_name=split_name,
    )

    lengths = {
        **numpy_lengths,
        "metadata_csv": len(metadata_csv),
    }

    if any(length != expected_cases for length in lengths.values()):
        raise ValueError(f"{split_name}: saved lengths are not aligned: {lengths}")

    if timestamps.shape != (expected_cases, n_timepoints):
        raise ValueError(
            f"{split_name}: timestamps has shape {timestamps.shape}; expected "
            f"{(expected_cases, n_timepoints)}."
        )

    observed_partitions = sorted(np.unique(partition_id).astype(int).tolist())
    if observed_partitions != sorted(expected_partitions):
        raise ValueError(
            f"{split_name}: expected partitions {sorted(expected_partitions)}, "
            f"found {observed_partitions}."
        )

    # Compare selected rows from the CSV and .npy sidecars.
    sample_indices = np.unique(
        np.linspace(0, expected_cases - 1, num=min(20, expected_cases), dtype=int)
    )

    csv_sample = metadata_csv.iloc[sample_indices].reset_index(drop=True)

    if not np.array_equal(
        csv_sample["partition_id"].to_numpy(dtype=np.int8),
        partition_id[sample_indices],
    ):
        raise ValueError(f"{split_name}: partition_id CSV/NPY mismatch.")

    if not np.array_equal(
        csv_sample["HARPNUM"].to_numpy(dtype=np.int64),
        harpnum[sample_indices],
    ):
        raise ValueError(f"{split_name}: HARPNUM CSV/NPY mismatch.")

    csv_start = csv_sample["start_time"].to_numpy(dtype="datetime64[ns]")
    csv_end = csv_sample["end_time"].to_numpy(dtype="datetime64[ns]")

    if not np.array_equal(csv_start, start_time[sample_indices]):
        raise ValueError(f"{split_name}: start_time CSV/NPY mismatch.")
    if not np.array_equal(csv_end, end_time[sample_indices]):
        raise ValueError(f"{split_name}: end_time CSV/NPY mismatch.")
    if not np.array_equal(timestamps[sample_indices, 0], csv_start):
        raise ValueError(f"{split_name}: timestamps first column mismatch.")
    if not np.array_equal(timestamps[sample_indices, -1], csv_end):
        raise ValueError(f"{split_name}: timestamps last column mismatch.")

    if not np.array_equal(
        csv_sample["source_file"].astype(str).to_numpy(),
        source_file[sample_indices].astype(str),
    ):
        raise ValueError(f"{split_name}: source_file CSV/NPY mismatch.")

    summary = {
        "split": split_name,
        "lengths": lengths,
        "X_shape": tuple(int(v) for v in X.shape),
        "y_dtype": str(y.dtype),
        "class_counts": class_count_dict(y),
        "partitions": observed_partitions,
        "unique_HARPNUM": int(np.unique(harpnum).size),
        "start_time_min": str(start_time.min()),
        "end_time_max": str(end_time.max()),
        "timestamps_shape": tuple(int(v) for v in timestamps.shape),
        "timestamp_cadence_minutes": TIME_STEP_MINUTES,
        "source_file_dtype": str(source_file.dtype),
        "csv_sidecar_sample_check": "passed",
    }

    del X, y, partition_id, harpnum, timestamps, start_time, end_time, source_file, metadata_csv
    gc.collect()

    return summary


train_alignment_summary = validate_saved_split(
    X_path=X_train_out_path,
    y_path=y_train_out_path,
    metadata_csv_path=train_metadata_csv_path,
    sidecar_paths=train_sidecar_paths,
    expected_cases=n_train,
    expected_partitions=TRAIN_PARTITIONS,
    split_name="train",
)

test_alignment_summary = validate_saved_split(
    X_path=X_test_out_path,
    y_path=y_test_out_path,
    metadata_csv_path=test_metadata_csv_path,
    sidecar_paths=test_sidecar_paths,
    expected_cases=n_test,
    expected_partitions=TEST_PARTITIONS,
    split_name="test",
)

print(json.dumps(train_alignment_summary, indent=2))
print(json.dumps(test_alignment_summary, indent=2))


## 8. Save build summary

In [ ]:
build_summary = {
    "project": "SWAN-SF solar flare classification",
    "notebook_purpose": (
        "Memory-efficient train/test split, five-feature train-only standardization, "
        "and aligned metadata sidecar creation"
    ),
    "input_directory": str(DATA_DIR),
    "output_directory": str(OUT_DIR),
    "train_partitions": TRAIN_PARTITIONS,
    "test_partitions": TEST_PARTITIONS,
    "selected_features": SELECTED_FEATURES,
    "selected_feature_indices_in_source_tensor": selected_feature_indices.tolist(),
    "source_feature_count": len(reference_feature_names),
    "x_key": X_KEY,
    "y_key": Y_KEY,
    "chunk_size": CHUNK_SIZE,
    "source_tensor_format": "(n_cases, n_timepoints, n_all_features)",
    "saved_aeon_tensor_format": "(n_cases, n_selected_features, n_timepoints)",
    "n_train": n_train,
    "n_test": n_test,
    "n_timepoints": n_timepoints,
    "n_selected_features": n_selected_features,
    "standardization": {
        "method": "feature-wise z-score",
        "fit_on": "train partitions only",
        "mean_axes_source_format": "axis=(0, 1), over cases and timepoints",
        "std_axes_source_format": "axis=(0, 1), over cases and timepoints",
        "zero_std_feature_count": int(zero_std_mask.sum()),
    },
    "metadata_sidecars": {
        "fields": [
            "partition_id",
            "HARPNUM",
            "timestamps",
            "start_time",
            "end_time",
            "source_file",
        ],
        "alignment_rule": (
            "Sidecar row i corresponds to X[i] and y[i] in the same split."
        ),
        "timestamps_shape_per_split": "(n_cases, n_timepoints)",
        "timestamp_cadence_minutes": TIME_STEP_MINUTES,
        "included_in_model_tensor": False,
        "standardized": False,
    },
    "raw_partition_summaries": partition_summaries,
    "combined_class_counts": combined_class_counts,
    "saved_train_tensor_summary": train_saved_summary,
    "saved_test_tensor_summary": test_saved_summary,
    "saved_train_alignment_summary": train_alignment_summary,
    "saved_test_alignment_summary": test_alignment_summary,
    "output_files": {
        "X_train_standardized_aeon": str(X_train_out_path),
        "y_train": str(y_train_out_path),
        "train_metadata_csv": str(train_metadata_csv_path),
        "train_sidecars": {
            key: str(path) for key, path in train_sidecar_paths.items()
        },
        "X_test_standardized_aeon": str(X_test_out_path),
        "y_test": str(y_test_out_path),
        "test_metadata_csv": str(test_metadata_csv_path),
        "test_sidecars": {
            key: str(path) for key, path in test_sidecar_paths.items()
        },
        "scaler_params": str(scaler_path),
        "selected_feature_names": str(feature_names_path),
    },
    "important_note": (
        "Metadata identifiers are saved as aligned sidecars and are not used as "
        "model features. Partition 4 is not used to fit the scaler."
    ),
}

build_summary_path = OUT_DIR / "five_feature_standardization_metadata.json"

with open(build_summary_path, "w") as f:
    json.dump(build_summary, f, indent=2, default=str)

print("Saved build summary:", build_summary_path)


## 9. Loading the final dataset later

Use memory mapping for the large tensors. The metadata arrays are one-dimensional and share the same row order.


In [ ]:
# Model arrays
X_train = np.load(
    OUT_DIR / "X_train_standardized_aeon.npy",
    mmap_mode="r",
)
y_train = np.load(
    OUT_DIR / "y_train.npy",
    mmap_mode="r",
)
X_test = np.load(
    OUT_DIR / "X_test_standardized_aeon.npy",
    mmap_mode="r",
)
y_test = np.load(
    OUT_DIR / "y_test.npy",
    mmap_mode="r",
)

# Aligned train metadata
train_partition_id = np.load(
    OUT_DIR / "train_partition_id.npy",
    mmap_mode="r",
)
train_harpnum = np.load(
    OUT_DIR / "train_HARPNUM.npy",
    mmap_mode="r",
)
train_timestamps = np.load(
    OUT_DIR / "train_timestamps.npy",
    mmap_mode="r",
)
train_start_time = np.load(
    OUT_DIR / "train_start_time.npy",
    mmap_mode="r",
)
train_end_time = np.load(
    OUT_DIR / "train_end_time.npy",
    mmap_mode="r",
)
train_source_file = np.load(
    OUT_DIR / "train_source_file.npy",
    mmap_mode="r",
)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_test:", X_test.shape, X_test.dtype)
print("y_test:", y_test.shape, y_test.dtype)

print("\nFirst aligned training case")
print("partition_id:", train_partition_id[0])
print("HARPNUM:", train_harpnum[0])
print("timestamps shape:", train_timestamps.shape)
print("first timestamp:", train_timestamps[0, 0])
print("last timestamp:", train_timestamps[0, -1])
print("start_time:", train_start_time[0])
print("end_time:", train_end_time[0])
print("source_file:", train_source_file[0])


## 10. Cleanup

In [ ]:
objects_to_delete = [
    "X_train",
    "y_train",
    "X_test",
    "y_test",
    "train_partition_id",
    "train_harpnum",
    "train_timestamps",
    "train_start_time",
    "train_end_time",
    "train_source_file",
]

for name in objects_to_delete:
    if name in globals():
        del globals()[name]

gc.collect()
print("Cleanup complete.")


In [ ]:
from pathlib import Path
import os

print("Configured output directory:")
print(OUT_DIR)

print("\nDirectory exists:", OUT_DIR.exists())

if OUT_DIR.exists():
    print("\nSaved files:")
    for path in sorted(OUT_DIR.iterdir()):
        size_gb = path.stat().st_size / (1024 ** 3)
        print(f"{path.name:45s} {size_gb:.3f} GB")
else:
    print("\nThe output folder does not exist.")